# 06 — Lamfalussy Ground Truth Validation

Validazione empirica del classificatore Lamfalussy (notebook 04) su atti del settore
finanziario UE il cui livello è una classificazione istituzionale **dichiarata**, non inferita.

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Testo articolo + full doc | LLM: classifica provision in L1–L4 | `segments_lamfalussy_gt.csv` |
| **B** | Classificazioni per articolo | Entropia normalizzata per articolo | `nodes_lamfalussy_gt.csv` |
| **C** | Entropia per articolo | Aggregazione → score ibridità per atto | `hybridity_gt.csv` |
| **D** | Score + ground truth | Confronto predicted vs declared level | report + `ground_truth_comparison.png` |

## 0. Configurazione

**Modifica solo questa cella.**

In [4]:
MATERIA_NAME = "ground_truth_validation"

# ── Modello ────────────────────────────────────────────────────────────────────
LLM_MODEL          = "gpt-4.1-mini"

# ── Parametri API ─────────────────────────────────────────────────────────────
LLM_MAX_TOKENS     = 2000
LLM_DELAY_SECONDS  = 0.3
LLM_MAX_RETRIES    = 3
LLM_RETRY_DELAY    = 5.0

# ── Parallelismo ──────────────────────────────────────────────────────────────
MAX_WORKERS        = 5

# ── Checkpoint ────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 50

# ── Contesto documento ────────────────────────────────────────────────────────
DOC_CONTEXT_MAX_CHARS = 40_000

# ── Ground truth acts ─────────────────────────────────────────────────────────
GROUND_TRUTH_ACTS = [
    {"celex": "32014L0065", "declared_level": "L1", "name": "MiFID II"},
    {"celex": "32014R0600", "declared_level": "L1", "name": "MiFIR"},
    {"celex": "32017R0565", "declared_level": "L2", "name": "MiFID II Delegated Regulation 2017/565"},
]

## 1. Import e Percorsi

In [5]:
import os, re, json, math, time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

# Input: cache globale full text (prodotta dal notebook 03)
NODES_TEXTS_FILE        = os.path.join('..', 'data', 'processed', 'nodes_texts.csv')
NODES_LIGHT_FILE        = os.path.join('..', 'data', 'processed', 'nodes_light.csv')

# Output
SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy_gt.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_gt_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy_gt.csv')
HYBRIDITY_FILE          = os.path.join(output_path, 'hybridity_gt.csv')

# Prompt template — stesso file usato dal notebook 04
PROMPT_FILE = os.path.join('..', 'notebooks', 'prompt.txt')

# ── Costanti Lamfalussy (identiche al notebook 04) ────────────────────────────
LAMFALUSSY_LEVELS = [
    {
        'key':  'L1',
        'name': 'Level 1 — Framework principles',
        'label': 'level_1',
        'description': (
            'Level 1 legislation sets out the core framework principles and defines essential features, '
            'as adopted by the European Parliament and Council in the co-decision procedure. '
            'Level 1 includes, for example, the objectives of the regulation; its scope and key definitions; '
            'the main rights and obligations; the institutional framework; core choices regarding harmonization '
            'between national and European levels; and the delegation clauses empowering the Commission or '
            'regulatory agencies to adopt implementing measures. These elements reflect fundamental political '
            'choices and are intended to be stable and enduring.'
        ),
    },
    {
        'key':  'L2',
        'name': 'Level 2 — Operational rules',
        'label': 'level_2',
        'description': (
            'Level 2 legislation translates Level 1 principles into operational rules by specifying their '
            'technical content and modes of application, adopted by the Commission via delegated acts, '
            'implementing acts, or technical standards (RTS/ITS) drafted by the ESAs. Level 2 includes, '
            'for example, detailed procedural rules for applying Level 1 obligations; specifications of '
            'quantitative thresholds and benchmarks; technical formats and data standards; and timelines for '
            'compliance. Level 2 is designed to be flexible, allowing continuous technical adjustments without '
            'reopening primary legislation.'
        ),
    },
    {
        'key':  'L3',
        'name': 'Level 3 — Supervisory convergence',
        'label': 'level_3',
        'description': (
            'Level 3 consists of measures issued by committees of national supervisors — now transformed into '
            'the European supervisory authorities (EBA, ESMA, EIOPA) — responsible for advising the Commission '
            'on Level 1 and Level 2 acts and for issuing guidelines on the implementation of the rules. '
            'Level 3 includes guidelines, recommendations, opinions, Q&As, and supervisory convergence tools '
            'that do not have binding legal force but provide practical direction on how to apply the rules '
            'uniformly at national level.'
        ),
    },
    {
        'key':  'L4',
        'name': 'Level 4 — Enforcement',
        'label': 'level_4',
        'description': (
            'Level 4 concerns the enforcement and monitoring of compliance with EU rules by national governments '
            'and competent authorities, with a stronger role for the Commission in ensuring correct application. '
            'Level 4 includes infringement proceedings, enforcement actions, compliance checks, and peer reviews '
            'aimed at verifying that Member States are correctly implementing and applying Level 1 and Level 2 '
            'legislation.'
        ),
    },
]

LAMF_KEYS     = [l['key']    for l in LAMFALUSSY_LEVELS]   # ['L1','L2','L3','L4']
LAMF_COLS     = [f'lamf_{k}' for k in LAMF_KEYS]           # ['lamf_L1',...,'lamf_L4']
LABEL_TO_KEY  = {l['label']: l['key'] for l in LAMFALUSSY_LEVELS}  # 'level_1' → 'L1'

GT_CELEXES = {act['celex'] for act in GROUND_TRUTH_ACTS}

print(f"Materia:      {MATERIA_NAME}")
print(f"Modello:      {LLM_MODEL}")
print(f"Livelli:      {LAMF_KEYS}")
print(f"Output:       {output_path}")
print(f"Ground truth: {[a['celex'] for a in GROUND_TRUTH_ACTS]}")

Materia:      ground_truth_validation
Modello:      gpt-4.1-mini
Livelli:      ['L1', 'L2', 'L3', 'L4']
Output:       ..\data\output\ground_truth_validation
Ground truth: ['32014L0065', '32014R0600', '32017R0565']


## 2. Fetch Testi EUR-Lex (solo atti non in cache)

Controlla quali CELEX ground truth mancano da `nodes_texts.csv`.
Per quelli assenti, scarica il testo da EUR-Lex usando le stesse funzioni del notebook 03
e aggiorna la cache globale — così la cella successiva li troverà.

In [6]:
from bs4 import BeautifulSoup
import eurlex

# ── Costanti fetch (identiche al notebook 03) ─────────────────────────────────
_DELAY_SECONDS    = 0.7
_TIMEOUT          = 20
_MAX_TITLE_CHARS  = 500
_SEGMENT_SPLIT    = 800

_PREAMBLE_END_MARKERS = [
    'HAVE ADOPTED THIS REGULATION:',
    'HAS ADOPTED THIS REGULATION:',
    'HAVE ADOPTED THIS DIRECTIVE:',
    'HAS ADOPTED THIS DIRECTIVE:',
    'HAVE ADOPTED THIS DECISION:',
    'HAS ADOPTED THIS DECISION:',
    'HAVE ADOPTED THIS FRAMEWORK DECISION:',
    'HAVE ADOPTED THIS RECOMMENDATION:',
    'HEREBY DECIDES:',
    'HAS DECIDED AS FOLLOWS:',
    'HEREBY RECOMMENDS:',
    'IS OF THE OPINION THAT:',
    'HAVE AGREED AS FOLLOWS:',
    'HAVE DECIDED AS FOLLOWS:',
]

_TEXT_COLS = [
    'title', 'preamble', 'articles', 'annexes',
    'full_text', 'segments', 'n_segments',
    'sections_found', 'text_status', 'text_length',
]


# ── Funzioni di estrazione (copiate dal notebook 03) ─────────────────────────

def _fetch_eurlex_html(celex):
    try:
        html = eurlex.get_html_by_celex_id(str(celex).strip(), language='en')
        if not html or len(html) < 500:
            return None, 'not_found'
        if not any(t in html for t in ['eli-subdivision', 'oj-doc-ti', 'doc-ti']):
            return None, 'no_structure'
        return html, 'ok'
    except Exception as e:
        err = str(e).lower()
        if 'timeout' in err:                   return None, 'timeout'
        if '404' in err or 'not found' in err: return None, 'not_found'
        return None, 'error'


def _extract_title(soup):
    for css in ['oj-doc-ti', 'doc-ti']:
        tags = soup.find_all('p', class_=css)
        if not tags:
            continue
        parts = []
        for tag in tags:
            t = tag.get_text(separator=' ', strip=True)
            if t.startswith(('ANNEX', 'SCHEDULE', 'APPENDIX')):
                break
            parts.append(t)
        if parts:
            return ' '.join(parts)[:_MAX_TITLE_CHARS]
    tag = soup.find('title')
    if tag:
        t = re.sub(r'\s*[-–|]\s*EUR-Lex.*$', '',
                   tag.get_text(strip=True), flags=re.IGNORECASE)
        if len(t) > 20:
            return t[:_MAX_TITLE_CHARS]
    return None


def _extract_preamble(soup):
    subdivs = soup.find_all(class_='eli-subdivision')
    if not subdivs:
        return []
    testo = re.sub(r'\s+', ' ', subdivs[0].get_text(separator=' ', strip=True)).strip()
    for marker in _PREAMBLE_END_MARKERS:
        idx = testo.find(marker)
        if idx != -1:
            testo = testo[:idx].strip()
            break
    if len(testo) < 50:
        return []
    parts = re.split(r'\((\d+)\)\s+', testo)
    segs = []
    if len(parts) <= 1:
        segs.append({'tipo': 'preambolo', 'identificatore': '0', 'testo': testo})
        return segs
    header = parts[0].strip()
    if header and len(header) >= 30:
        segs.append({'tipo': 'preambolo_header', 'identificatore': '0', 'testo': header})
    i = 1
    while i < len(parts) - 1:
        num, corpo = parts[i].strip(), parts[i+1].strip()
        if corpo and len(corpo) >= 20:
            segs.append({'tipo': 'considerando', 'identificatore': num, 'testo': corpo})
        i += 2
    return segs


def _extract_articles(soup):
    divs = [d for d in soup.find_all(class_='eli-subdivision')
            if re.match(r'^art_\d+', d.get('id', ''))]
    if not divs:
        divs = soup.find_all(class_='eli-subdivision', attrs={'data-section': 'article'})
    if not divs:
        divs = [d for d in soup.find_all(class_='eli-subdivision')[1:]
                if re.match(r'^Article\s+\d+',
                            d.get_text(separator=' ', strip=True)[:100], re.IGNORECASE)]
    segs = []
    for div in divs:
        testo = re.sub(r'\s+', ' ', div.get_text(separator=' ', strip=True)).strip()
        if len(testo) < 10:
            continue
        m   = re.match(r'Article\s+(\d+[a-z]?)', testo, re.IGNORECASE)
        num = m.group(1) if m else re.sub(r'^art_', '', div.get('id', str(len(segs)+1)))
        segs.append({'tipo': 'articolo', 'identificatore': num, 'testo': testo})
    return segs


def _extract_annexes(soup):
    patterns = [r'^anx_', r'^ann_', r'^annex']
    divs = [d for d in soup.find_all(class_='eli-subdivision')
            if any(re.match(p, d.get('id', '').lower()) for p in patterns)]
    if not divs:
        divs = [d for d in soup.find_all(class_='eli-subdivision')
                if re.match(r'^(ANNEX|APPENDIX|SCHEDULE)\b',
                            d.get_text(separator=' ', strip=True)[:60], re.IGNORECASE)]
    segs = []
    for div in divs:
        testo = re.sub(r'\s+', ' ', div.get_text(separator=' ', strip=True)).strip()
        if len(testo) < 10:
            continue
        m     = re.match(r'(?:ANNEX|APPENDIX|SCHEDULE)\s+([IVXivx\d]+[A-Za-z]?)',
                         testo[:80], re.IGNORECASE)
        ident = m.group(1).upper() if m else str(len(segs)+1)
        segs.append({'tipo': 'allegato', 'identificatore': ident, 'testo': testo})
    return segs


def _split_article(seg):
    testo = seg['testo']
    if len(testo) <= _SEGMENT_SPLIT:
        return [seg]
    paragraphs = [p.strip() for p in re.split(
        r'(?=(?:^|\s)(?:\d+\.\s|\([a-z]\)\s|\([ivx]+\)\s))', testo)
        if p.strip() and len(p.strip()) > 30]
    if len(paragraphs) <= 1:
        return [seg]
    return [{'tipo': seg['tipo'],
             'identificatore': f"{seg['identificatore']}_p{i}",
             'testo': para}
            for i, para in enumerate(paragraphs, start=1)]


def _build_segments(preamble_segs, article_segs, annex_segs):
    all_segs = list(preamble_segs)
    for seg in article_segs:
        all_segs.extend(_split_article(seg))
    all_segs.extend(annex_segs)
    for i, seg in enumerate(all_segs):
        seg['segment_id'] = i
    return all_segs


def _extract_all_sections(celex):
    empty = {'title': None, 'preamble': None, 'articles': None, 'annexes': None,
             'full_text': None, 'segments': None, 'n_segments': 0,
             'sections_found': '', 'text_status': None, 'text_length': 0}
    html, status = _fetch_eurlex_html(celex)
    if status != 'ok':
        return {**empty, 'text_status': status}
    try:
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return {**empty, 'text_status': 'parse_error'}

    title        = _extract_title(soup)
    preamble_s   = _extract_preamble(soup)
    article_s    = _extract_articles(soup)
    annex_s      = _extract_annexes(soup)

    preamble_txt = ' \n '.join(s['testo']   for s in preamble_s) if preamble_s else None
    articles_txt = ' \n\n '.join(s['testo'] for s in article_s)  if article_s  else None
    annexes_txt  = ' \n\n '.join(s['testo'] for s in annex_s)    if annex_s    else None
    segments     = _build_segments(preamble_s, article_s, annex_s)

    sections_found = ','.join(filter(None, [
        'title'    if title       else None,
        'preamble' if preamble_s  else None,
        'articles' if article_s   else None,
        'annexes'  if annex_s     else None,
    ]))
    parti = []
    if title:        parti.append(f"[TITLE] {title}")
    if preamble_txt: parti.append(f"[PREAMBLE] {preamble_txt}")
    if articles_txt: parti.append(f"[ARTICLES] {articles_txt}")
    if annexes_txt:  parti.append(f"[ANNEXES] {annexes_txt}")
    if not parti:
        return {**empty, 'text_status': 'no_content'}

    full_text = ' \n\n '.join(parti)
    return {
        'title': title, 'preamble': preamble_txt,
        'articles': articles_txt, 'annexes': annexes_txt,
        'full_text': full_text,
        'segments': json.dumps(segments, ensure_ascii=False),
        'n_segments': len(segments),
        'sections_found': sections_found,
        'text_status': 'ok',
        'text_length': len(full_text),
    }


# ── Controlla quali CELEX mancano dalla cache globale ─────────────────────────
global_cache = pd.read_csv(NODES_TEXTS_FILE, low_memory=False) \
               if os.path.exists(NODES_TEXTS_FILE) else pd.DataFrame(columns=['Label'] + _TEXT_COLS)

cache_celex_col = 'Label' if 'Label' in global_cache.columns else 'Id'
cached_celexes  = set(global_cache[cache_celex_col].dropna().astype(str))

missing = [act for act in GROUND_TRUTH_ACTS if act['celex'] not in cached_celexes]
print(f"Cache globale: {len(global_cache):,} CELEX")
print(f"GT acts in cache: {len(GROUND_TRUTH_ACTS) - len(missing)}/{len(GROUND_TRUTH_ACTS)}")
print(f"Da scaricare:     {len(missing)}")
for act in missing:
    print(f"  - {act['celex']}  ({act['name']})")

# ── Fetch degli atti mancanti ─────────────────────────────────────────────────
if missing:
    print(f"\nInizio fetch ({len(missing)} atti)...")
    new_rows = []
    for act in missing:
        celex = act['celex']
        print(f"  Fetching {celex} ({act['name']})...", end=' ', flush=True)
        result = _extract_all_sections(celex)
        status = result['text_status']
        n_segs = result['n_segments']
        print(f"→ {status}  ({n_segs} segmenti)")
        if status == 'ok':
            # Aggiungi metadati minimi per compatibilità con nodes_texts.csv
            nodes_light = pd.read_csv(NODES_LIGHT_FILE)
            meta = nodes_light[nodes_light['celex'] == celex]
            new_row = {
                'Id':    celex,
                'Label': celex,
            }
            if len(meta):
                m = meta.iloc[0]
                new_row.update({
                    'Year':            m.get('year_final', ''),
                    'LegalType':       m.get('legal_type_normalized', ''),
                    'Era':             m.get('era', ''),
                    'Decade':          m.get('decade', ''),
                    'PipelineLevel':   '',
                    'SeedConceptCount': '',
                    'titolo':          '',
                })
            new_row.update(result)
            new_rows.append(new_row)
        time.sleep(_DELAY_SECONDS)

    # ── Aggiorna cache globale ────────────────────────────────────────────────
    if new_rows:
        new_df       = pd.DataFrame(new_rows)
        global_cache = pd.concat([global_cache, new_df], ignore_index=True)
        global_cache.to_csv(NODES_TEXTS_FILE, index=False)
        n_ok = sum(r['text_status'] == 'ok' for r in new_rows)
        print(f"\n✓ Cache aggiornata: +{n_ok} atti  (totale: {len(global_cache):,} CELEX)")
        print(f"  Salvata in: {NODES_TEXTS_FILE}")
    else:
        print("\n⚠ Nessun atto recuperato con successo.")
else:
    print("\n✓ Tutti gli atti ground truth già in cache — nessun fetch necessario.")


Cache globale: 1,794 CELEX
GT acts in cache: 0/3
Da scaricare:     3
  - 32014L0065  (MiFID II)
  - 32014R0600  (MiFIR)
  - 32017R0565  (MiFID II Delegated Regulation 2017/565)

Inizio fetch (3 atti)...
  Fetching 32014L0065 (MiFID II)... → ok  (1259 segmenti)
  Fetching 32014R0600 (MiFIR)... → ok  (627 segmenti)
  Fetching 32017R0565 (MiFID II Delegated Regulation 2017/565)... → ok  (940 segmenti)

✓ Cache aggiornata: +3 atti  (totale: 1,797 CELEX)
  Salvata in: ..\data\processed\nodes_texts.csv


## 3. Caricamento Dati dalla Cache Globale

Legge `nodes_texts.csv` (cache aggiornata dalla cella precedente).
Filtra solo gli atti in `GROUND_TRUTH_ACTS`. Se un CELEX non è presente, logga un warning.

In [7]:
nodes_all = pd.read_csv(NODES_TEXTS_FILE)
print(f"Cache globale: {len(nodes_all):,} nodi")
print(f"Colonne:       {nodes_all.columns.tolist()}")

# ── Filtra solo gli atti ground truth ─────────────────────────────────────────
# La colonna CELEX può stare in 'Label' o 'Id'
celex_col = 'Label' if 'Label' in nodes_all.columns else 'Id'

found_celexes = set()
skipped_celexes = set()
nodes_gt_rows = []

for act in GROUND_TRUTH_ACTS:
    celex = act['celex']
    mask = nodes_all[celex_col] == celex
    matches = nodes_all[mask]
    # Se ci sono duplicati, prendi il primo con text_status == 'ok', altrimenti il primo
    ok_matches = matches[matches.get('text_status', pd.Series(['ok']*len(matches))) == 'ok']
    chosen = ok_matches.head(1) if len(ok_matches) else matches.head(1)
    if len(chosen):
        found_celexes.add(celex)
        nodes_gt_rows.append(chosen.iloc[0])
    else:
        skipped_celexes.add(celex)
        print(f"WARNING: CELEX {celex} ({act['name']}) non presente nella cache — skippato")

nodes_gt = pd.DataFrame(nodes_gt_rows) if nodes_gt_rows else pd.DataFrame()
print(f"\nAtti trovati:  {len(found_celexes)}  →  {sorted(found_celexes)}")
print(f"Atti skippati: {len(skipped_celexes)}  →  {sorted(skipped_celexes)}")

if nodes_gt.empty:
    print("\n⚠ Nessun atto ground truth trovato nella cache.")
    print("  Aggiungere i CELEX al corpus e rieseguire il notebook 03 per popolare nodes_texts.csv.")
else:
    for _, row in nodes_gt.iterrows():
        celex = str(row.get(celex_col, ''))
        n_seg = row.get('n_segments', '?')
        status = row.get('text_status', '?')
        title  = str(row.get('title', ''))[:80]
        print(f"  {celex}  status={status}  n_segments={n_seg}  title={title}")

Cache globale: 1,797 nodi
Colonne:       ['Id', 'Label', 'Year', 'LegalType', 'Era', 'Decade', 'PipelineLevel', 'SeedConceptCount', 'title', 'preamble', 'articles', 'annexes', 'full_text', 'segments', 'n_segments', 'sections_found', 'text_status', 'text_length', 'titolo']

Atti trovati:  3  →  ['32014L0065', '32014R0600', '32017R0565']
Atti skippati: 0  →  []
  32014L0065  status=ok  n_segments=1259  title=DIRECTIVE 2014/65/EU OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 15 May 201
  32014R0600  status=ok  n_segments=627  title=REGULATION (EU) No 600/2014 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 15 
  32017R0565  status=ok  n_segments=940  title=COMMISSION DELEGATED REGULATION (EU) 2017/565 of 25 April 2016 supplementing Dir


## 3. Segmentazione Articoli

In [8]:
rows = []
FULL_TEXTS = {}

if not nodes_gt.empty:
    for _, node in nodes_gt.iterrows():
        celex = str(node.get(celex_col, node.get('Id', '')))
        title = str(node.get('title', ''))
        raw   = node.get('segments', '')

        # Full text per contesto documento
        ft = str(node.get('full_text', '') or '')
        if ft and ft != 'nan':
            FULL_TEXTS[celex] = ft

        if pd.isna(raw) or not str(raw).strip() or str(raw) in ('nan', '[]'):
            print(f"WARNING: CELEX {celex} — segmenti non disponibili, skippato")
            continue

        try:
            segs = json.loads(str(raw))
        except (json.JSONDecodeError, ValueError):
            print(f"WARNING: CELEX {celex} — errore parsing segmenti, skippato")
            continue

        seen = set()
        n_art = 0
        for i, s in enumerate(segs):
            if s.get('tipo') != 'articolo':
                continue
            testo = str(s.get('testo', '')).strip()
            if len(testo) < 30:
                continue
            idf    = str(s.get('identificatore', i))
            seg_id = f"{celex}__{idf}"
            if seg_id in seen:
                continue
            seen.add(seg_id)
            n_art += 1
            rows.append({
                'segment_id':     seg_id,
                'celex':          celex,
                'node_id':        str(node.get('Id', celex)),
                'title_atto':     title,
                'tipo':           s.get('tipo'),
                'identificatore': idf,
                'testo':          testo,
            })
        print(f"  {celex}: {n_art} articoli estratti")

articles_df = pd.DataFrame(rows)

print(f"\nArticoli totali: {len(articles_df):,}")
print(f"Atti coinvolti:  {articles_df['celex'].nunique() if not articles_df.empty else 0}")
print(f"Full text:       {len(FULL_TEXTS)} atti")

if articles_df.empty:
    print("\n⚠ Nessun articolo disponibile. Esecuzione interrotta — eseguire il notebook dopo")
    print("  aver aggiunto gli atti ground truth al corpus e rieseguito il notebook 03.")

  32014L0065: 1079 articoli estratti
  32014R0600: 565 articoli estratti
  32017R0565: 765 articoli estratti

Articoli totali: 2,409
Atti coinvolti:  3
Full text:       3 atti


## 4. Fase A — Classificazione Lamfalussy (LLM)

Identica al notebook 04: per ogni articolo, il testo è tokenizzato e l'LLM
classifica ogni provision come `level_1`–`level_4`. Le percentuali L1–L4
emergono dal conteggio dei token per livello.

Il prompt template è lo stesso file `prompt.txt` usato dal notebook 04.

In [9]:
# ── Prompt template ───────────────────────────────────────────────────────────
with open(PROMPT_FILE, encoding='utf-8') as _f:
    PROMPT_TEMPLATE = _f.read()

print(f'Prompt caricato da: {PROMPT_FILE}  ({len(PROMPT_TEMPLATE)} caratteri)')
for ph in ['{{DOCUMENT_CONTEXT}}', '{{CHUNK_TEXT}}', '{{EXPECTED_UNITS}}', '{{DOCUMENT_NAME}}']:
    status = '✓' if ph in PROMPT_TEMPLATE else '✗ MANCANTE'
    print(f'  {status}  {ph}')


def tokenize_with_indices(text: str) -> tuple[list[str], str]:
    words = text.split()
    indexed = ' '.join(f'({i+1}){w}' for i, w in enumerate(words))
    return words, indexed


def build_prompt(doc_text: str, art_text: str, art_id: str, doc_name: str) -> str:
    _, indexed_art = tokenize_with_indices(art_text)
    chunk    = f"[ARTICLE {art_id}]\n[ARTICLE_TEXT]\n{indexed_art}"
    unit_id  = 'i000001'
    expected = json.dumps({unit_id: {'article_id': str(art_id), 'comma_id': None}}, indent=2)
    ctx      = doc_text[:DOC_CONTEXT_MAX_CHARS] if doc_text else '(not available)'
    return (PROMPT_TEMPLATE
        .replace('{{DOCUMENT_CONTEXT}}', ctx)
        .replace('{{CHUNK_TEXT}}',       chunk)
        .replace('{{EXPECTED_UNITS}}',   expected)
        .replace('{{DOCUMENT_NAME}}',    doc_name))


def parse_response(response_text: str, art_text: str) -> dict | None:
    try:
        clean = re.sub(r'^```[a-z]*\n?', '', response_text.strip())
        clean = re.sub(r'\n?```$', '', clean)
        data  = json.loads(clean)
    except json.JSONDecodeError:
        return None

    classes = data.get('classifications', {})
    if not classes:
        return None
    unit       = next(iter(classes.values()), {})
    provisions = unit.get('provisions', [])
    if not provisions:
        return None

    words, _ = tokenize_with_indices(art_text)
    total_tokens = len(words)

    counts = {k: 0 for k in LAMF_KEYS}
    counts['unassigned'] = 0
    evidence = []

    for p in provisions:
        start  = int(p.get('start', 1))
        end    = int(p.get('end', total_tokens))
        label  = str(p.get('label', 'unassigned'))
        reason = str(p.get('reason', ''))
        span   = max(0, end - start + 1)
        key    = LABEL_TO_KEY.get(label, 'unassigned')
        counts[key] = counts.get(key, 0) + span

        if key != 'unassigned' and span > 0:
            quote = ' '.join(words[start-1:end])
            if len(quote) > 200:
                quote = quote[:197] + '...'
            evidence.append({'testo': quote, 'layer': key, 'motivo': reason})

    total_labeled = sum(counts.get(k, 0) for k in LAMF_KEYS)
    if total_labeled == 0:
        return None

    pcts = {k: round(counts.get(k, 0) / total_labeled * 100, 2) for k in LAMF_KEYS}
    return {'pcts': pcts, 'evidence': evidence, 'provisions': provisions}


def call_llm(client, prompt: str) -> tuple[str, str]:
    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=LLM_MAX_TOKENS,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
            )
            return resp.choices[0].message.content, 'ok'
        except openai.RateLimitError:
            time.sleep(LLM_RETRY_DELAY * (attempt + 1))
        except Exception as e:
            if attempt == LLM_MAX_RETRIES - 1:
                return str(e), 'error'
            time.sleep(LLM_RETRY_DELAY)
    return 'max_retries_exceeded', 'error'


print('Funzioni Fase A definite.')
print(f'Livelli Lamfalussy: {LAMF_KEYS}')

Prompt caricato da: ..\notebooks\prompt.txt  (7149 caratteri)
  ✓  {{DOCUMENT_CONTEXT}}
  ✓  {{CHUNK_TEXT}}
  ✓  {{EXPECTED_UNITS}}
  ✓  {{DOCUMENT_NAME}}
Funzioni Fase A definite.
Livelli Lamfalussy: ['L1', 'L2', 'L3', 'L4']


In [10]:
%%time
if articles_df.empty:
    print("⚠ Nessun articolo da classificare — cella saltata.")
else:
    client = OpenAI()

    # ── Checkpoint ────────────────────────────────────────────────────────────
    done_ids = set()
    if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
        ckpt_df  = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
        done_ids = set(ckpt_df['segment_id'])
        print(f'Checkpoint: {len(done_ids):,} articoli già classificati.')
    else:
        ckpt_df = pd.DataFrame()
        print('Nessun checkpoint — si parte da zero.')

    todo_df = articles_df[~articles_df['segment_id'].isin(done_ids)].copy()
    print(f'Articoli da classificare: {len(todo_df):,}  |  già ok: {len(done_ids):,}')

    if len(todo_df) == 0:
        print('✓ Tutti i segmenti già classificati — si può passare alla Fase B.')

    ckpt_lock   = Lock()
    buffer      = []
    n_ok        = 0
    n_error     = 0
    n_processed = 0

    def process_article(seg: pd.Series) -> dict:
        celex  = seg['celex']
        art_id = seg['identificatore']
        testo  = seg['testo']
        prompt = build_prompt(
            doc_text = FULL_TEXTS.get(celex, ''),
            art_text = testo,
            art_id   = art_id,
            doc_name = celex,
        )
        time.sleep(LLM_DELAY_SECONDS)
        response_text, status = call_llm(client, prompt)

        parsed = None
        if status == 'ok':
            parsed = parse_response(response_text, testo)
            if parsed is None:
                status = 'parse_error'

        row = {
            'segment_id':     seg['segment_id'],
            'celex':          celex,
            'node_id':        seg['node_id'],
            'tipo':           seg['tipo'],
            'identificatore': art_id,
            'llm_status':     status,
            'evidence':       json.dumps(parsed['evidence'], ensure_ascii=False) if parsed else '[]',
            'provisions_raw': json.dumps(parsed['provisions'], ensure_ascii=False) if parsed else '[]',
        }
        for col, key in zip(LAMF_COLS, LAMF_KEYS):
            row[col] = round(parsed['pcts'][key], 2) if parsed else 0.0
        return row

    def save_checkpoint(new_rows):
        if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
            existing = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
        else:
            existing = pd.DataFrame()
        parts    = [p for p in [existing, pd.DataFrame(new_rows)] if not p.empty]
        combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        combined.to_csv(SEGMENTS_LAMF_CKPT_FILE, index=False)

    total        = len(todo_df)
    todo_records = [row for _, row in todo_df.iterrows()]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_article, seg): seg['segment_id'] for seg in todo_records}
        for future in as_completed(futures):
            row = future.result()
            with ckpt_lock:
                buffer.append(row)
                n_processed += 1
                if row['llm_status'] == 'ok':
                    n_ok += 1
                else:
                    n_error += 1
                if n_processed % CHECKPOINT_EVERY == 0 or n_processed == total:
                    save_checkpoint(buffer)
                    buffer.clear()
                    pct = n_processed / total * 100 if total else 100
                    print(f'  [{n_processed:>5}/{total}]  {pct:5.1f}%   ok: {n_ok}   errori: {n_error}')

    print()
    print('=' * 50)
    print(f'FASE A — ok: {n_ok:,}   errori: {n_error:,}')
    print('=' * 50)

Nessun checkpoint — si parte da zero.
Articoli da classificare: 2,409  |  già ok: 0
  [   50/2409]    2.1%   ok: 50   errori: 0
  [  100/2409]    4.2%   ok: 100   errori: 0
  [  150/2409]    6.2%   ok: 150   errori: 0
  [  200/2409]    8.3%   ok: 200   errori: 0
  [  250/2409]   10.4%   ok: 250   errori: 0
  [  300/2409]   12.5%   ok: 300   errori: 0
  [  350/2409]   14.5%   ok: 349   errori: 1
  [  400/2409]   16.6%   ok: 399   errori: 1
  [  450/2409]   18.7%   ok: 449   errori: 1
  [  500/2409]   20.8%   ok: 499   errori: 1
  [  550/2409]   22.8%   ok: 549   errori: 1
  [  600/2409]   24.9%   ok: 599   errori: 1
  [  650/2409]   27.0%   ok: 649   errori: 1
  [  700/2409]   29.1%   ok: 698   errori: 2
  [  750/2409]   31.1%   ok: 747   errori: 3
  [  800/2409]   33.2%   ok: 797   errori: 3
  [  850/2409]   35.3%   ok: 847   errori: 3
  [  900/2409]   37.4%   ok: 897   errori: 3
  [  950/2409]   39.4%   ok: 946   errori: 4
  [ 1000/2409]   41.5%   ok: 996   errori: 4
  [ 1050/2409]   

In [11]:
if articles_df.empty:
    print("⚠ Nessun articolo classificato — cella saltata.")
    segments_lamf = pd.DataFrame()
elif os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    segments_lamf = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    segments_lamf.to_csv(SEGMENTS_LAMF_FILE, index=False)

    ok_mask = segments_lamf['llm_status'] == 'ok'
    print(f'Salvato: {SEGMENTS_LAMF_FILE}')
    print(f'Totale: {len(segments_lamf):,}  |  ok: {ok_mask.sum():,}')
    print()
    print('Distribuzione media % per livello Lamfalussy (articoli ok):')
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
        mean_pct = segments_lamf.loc[ok_mask, col].mean()
        bar      = '█' * int(mean_pct / 2)
        print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")
else:
    segments_lamf = pd.DataFrame()
    print('⚠ Nessun checkpoint trovato.')

Salvato: ..\data\output\ground_truth_validation\segments_lamfalussy_gt.csv
Totale: 2,409  |  ok: 2,396

Distribuzione media % per livello Lamfalussy (articoli ok):
  Level 1 — Framework principles................  41.1%  ████████████████████
  Level 2 — Operational rules...................  42.4%  █████████████████████
  Level 3 — Supervisory convergence.............   0.8%  
  Level 4 — Enforcement.........................  15.7%  ███████


## 5. Fase B — Entropia per Articolo

Identica al notebook 04: entropia di Shannon normalizzata sulla distribuzione L1–L4.

$$H(a) = -\frac{\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)}{\log_2(4)}$$

In [12]:
def entropy_norm(row, lamf_cols):
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in lamf_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    rawH  = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(lamf_cols)) if len(lamf_cols) > 1 else 1.0
    return float(np.clip(rawH / maxH, 0, 1))


if segments_lamf.empty:
    print("⚠ Nessun segmento classificato — cella saltata.")
    art_df = pd.DataFrame()
else:
    seg_df = segments_lamf if not segments_lamf.empty else pd.read_csv(SEGMENTS_LAMF_FILE)
    art_df = seg_df[seg_df['llm_status'] == 'ok'].copy()

    for col in LAMF_COLS:
        if col not in art_df.columns:
            art_df[col] = 0.0

    art_df['entropy']      = art_df.apply(lambda r: entropy_norm(r, LAMF_COLS), axis=1)
    art_df['dominant_lamf'] = art_df[LAMF_COLS].idxmax(axis=1).str.replace('lamf_', '', regex=False)
    art_df['articolo_id']  = art_df['identificatore']

    art_df.to_csv(NODES_LAMFALUSSY_FILE, index=False)

    print(f'Salvato: {NODES_LAMFALUSSY_FILE}')
    print(f'Articoli: {len(art_df):,}  |  Atti: {art_df["celex"].nunique():,}')
    print(f'Entropia media: {art_df["entropy"].mean():.4f}  |  max: {art_df["entropy"].max():.4f}')
    print()
    print('Distribuzione media % per livello:')
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
        mean_pct = art_df[col].mean()
        bar      = '█' * int(mean_pct / 2)
        print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\ground_truth_validation\nodes_lamfalussy_gt.csv
Articoli: 2,396  |  Atti: 3
Entropia media: 0.0649  |  max: 0.7884

Distribuzione media % per livello:
  Level 1 — Framework principles................  41.1%  ████████████████████
  Level 2 — Operational rules...................  42.4%  █████████████████████
  Level 3 — Supervisory convergence.............   0.8%  
  Level 4 — Enforcement.........................  15.7%  ███████


## 6. Fase C — Aggregazione per Atto

Identica al notebook 04: score di ibridità = entropia della distribuzione L1-L4 **media**
degli articoli dell'atto. Differisce dalla media delle entropie per articolo.

In [13]:
LAMF_NAMES = {l['key']: l['name'] for l in LAMFALUSSY_LEVELS}


def entropy_of_mean(group, lamf_cols):
    eps       = 1e-9
    mean_vals = group[lamf_cols].mean().values.astype(float)
    s         = mean_vals.sum()
    if s < eps:
        return 0.0
    probs = mean_vals / s
    rawH  = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(lamf_cols))
    return float(np.clip(rawH / maxH, 0, 1))


def agg_atto(group):
    dom             = group['dominant_lamf'].mode()
    hyb_max_row     = group.nlargest(1, 'entropy')
    dominant_key    = dom.iloc[0] if len(dom) else ''
    return pd.Series({
        'hybridity_score':     entropy_of_mean(group, LAMF_COLS),
        'hybridity_std':       group['entropy'].std(),
        'hybridity_max':       group['entropy'].max(),
        'n_articles':          len(group),
        'dominant_lamf':       dominant_key,
        'dom':                 LAMF_NAMES.get(dominant_key, dominant_key),
        'dominant_lamf_pct':   (group['dominant_lamf'] == dominant_key).mean() * 100 if dominant_key else 0.0,
        'most_hybrid_article': hyb_max_row['articolo_id'].iloc[0] if len(hyb_max_row) else '',
        **{col: group[col].mean() for col in LAMF_COLS},
    })


if art_df.empty:
    print("⚠ Nessun articolo disponibile — cella saltata.")
    hybridity_df = pd.DataFrame()
else:
    hybridity_df = art_df.groupby('celex').apply(agg_atto).reset_index()

    hybridity_df.to_csv(HYBRIDITY_FILE, index=False)

    print(f'Salvato: {HYBRIDITY_FILE}')
    print(f'Atti analizzati: {len(hybridity_df):,}')
    print()
    for _, row in hybridity_df.iterrows():
        dist = {k: round(row[f'lamf_{k}'], 1) for k in LAMF_KEYS}
        print(f"  {row['celex']:<22}  dominant={row['dominant_lamf']}  H={row['hybridity_score']:.3f}  dist={dist}")

Salvato: ..\data\output\ground_truth_validation\hybridity_gt.csv
Atti analizzati: 3

  32014L0065              dominant=L1  H=0.797  dist={'L1': 43.9, 'L2': 34.0, 'L3': 1.1, 'L4': 21.0}
  32014R0600              dominant=L1  H=0.783  dist={'L1': 42.8, 'L2': 37.6, 'L3': 0.8, 'L4': 18.8}
  32017R0565              dominant=L2  H=0.632  dist={'L1': 35.8, 'L2': 57.9, 'L3': 0.5, 'L4': 5.8}


## 7. Confronto con Ground Truth

Per ogni atto classificato, confronta il livello dominante predetto con quello
dichiarato istituzionalmente. Un match (`✓`) costituisce validazione empirica
indiretta della pipeline.

In [14]:
if hybridity_df.empty:
    print("⚠ Nessun risultato disponibile — confronto non eseguibile.")
    print("  Aggiungere gli atti al corpus (notebook 03) e rieseguire.")
else:
    n_match = 0
    n_total = 0

    print("Ground Truth Validation — Classifier output vs. declared Lamfalussy level")
    print("=" * 70)

    for act in GROUND_TRUTH_ACTS:
        celex    = act['celex']
        declared = act['declared_level']

        if celex in skipped_celexes:
            print(f"⊘ {act['name']}")
            print(f"   CELEX {celex} non presente nel corpus — skippato")
            print()
            continue

        row = hybridity_df[hybridity_df['celex'] == celex]
        if row.empty:
            print(f"⊘ {act['name']}")
            print(f"   CELEX {celex} non presente nei risultati aggregati")
            print()
            continue

        predicted = row['dominant_lamf'].values[0]
        score     = row['hybridity_score'].values[0]
        dist      = {k: round(row[f'lamf_{k}'].values[0], 1) for k in LAMF_KEYS}
        n_arts    = int(row['n_articles'].values[0])

        match  = '✓' if predicted == declared else '✗'
        n_total += 1
        if predicted == declared:
            n_match += 1

        print(f"{match} {act['name']}  ({celex})")
        print(f"   Declared: {declared}  |  Predicted dominant: {predicted}  |  H: {score:.3f}  |  n_articles: {n_arts}")
        print(f"   Distribution: {dist}")
        print()

    if n_total > 0:
        acc = n_match / n_total * 100
        print("=" * 70)
        print(f"Accuracy: {n_match}/{n_total}  ({acc:.0f}%)")

Ground Truth Validation — Classifier output vs. declared Lamfalussy level
✓ MiFID II  (32014L0065)
   Declared: L1  |  Predicted dominant: L1  |  H: 0.797  |  n_articles: 1075
   Distribution: {'L1': np.float64(43.9), 'L2': np.float64(34.0), 'L3': np.float64(1.1), 'L4': np.float64(21.0)}

✓ MiFIR  (32014R0600)
   Declared: L1  |  Predicted dominant: L1  |  H: 0.783  |  n_articles: 563
   Distribution: {'L1': np.float64(42.8), 'L2': np.float64(37.6), 'L3': np.float64(0.8), 'L4': np.float64(18.8)}

✓ MiFID II Delegated Regulation 2017/565  (32017R0565)
   Declared: L2  |  Predicted dominant: L2  |  H: 0.632  |  n_articles: 758
   Distribution: {'L1': np.float64(35.8), 'L2': np.float64(57.9), 'L3': np.float64(0.5), 'L4': np.float64(5.8)}

Accuracy: 3/3  (100%)


## 8. Visualizzazione

Bar chart con distribuzione L1–L4 per ogni atto classificato,
con annotazione del livello dichiarato.

In [15]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (safe for all environments)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

LEVEL_COLORS = {
    'L1': '#2563eb',  # blue
    'L2': '#16a34a',  # green
    'L3': '#d97706',  # amber
    'L4': '#dc2626',  # red
}

if hybridity_df.empty:
    print("⚠ Nessun dato — visualizzazione non disponibile.")
else:
    # ── Costruisce tabella risultati ──────────────────────────────────────────
    result_rows = []
    for act in GROUND_TRUTH_ACTS:
        celex    = act['celex']
        declared = act['declared_level']
        row      = hybridity_df[hybridity_df['celex'] == celex]
        if row.empty:
            continue
        dist = {k: round(row[f'lamf_{k}'].values[0], 1) for k in LAMF_KEYS}
        result_rows.append({
            'name':      act['name'],
            'celex':     celex,
            'declared':  declared,
            'predicted': row['dominant_lamf'].values[0],
            'H':         row['hybridity_score'].values[0],
            **dist,
        })

    if not result_rows:
        print("⚠ Nessun atto classificato — grafico non prodotto.")
    else:
        # ── Tabella Markdown ──────────────────────────────────────────────────
        print("| Atto | Declared | Predicted | H | L1% | L2% | L3% | L4% | Match |")
        print("|---|---|---|---|---|---|---|---|---|")
        for r in result_rows:
            match = '✓' if r['predicted'] == r['declared'] else '✗'
            print(f"| {r['name']} | {r['declared']} | {r['predicted']} | {r['H']:.3f} "
                  f"| {r['L1']:.1f} | {r['L2']:.1f} | {r['L3']:.1f} | {r['L4']:.1f} | {match} |")
        print()

        # ── Bar chart ─────────────────────────────────────────────────────────
        n_acts  = len(result_rows)
        x       = np.arange(n_acts)
        width   = 0.18
        offsets = np.array([-1.5, -0.5, 0.5, 1.5]) * width

        fig, ax = plt.subplots(figsize=(max(8, n_acts * 3), 6))

        for j, key in enumerate(LAMF_KEYS):
            vals = [r[key] for r in result_rows]
            ax.bar(x + offsets[j], vals, width=width,
                   color=LEVEL_COLORS[key], label=key, alpha=0.85)

        # Linea divisoria e label "declared" sopra ogni gruppo
        for i, r in enumerate(result_rows):
            ax.axvline(x=i, color='black', linewidth=0.5, linestyle='--', alpha=0.3)
            ax.text(i, 101, f"declared: {r['declared']}",
                    ha='center', fontsize=8, color='#374151')

        # Stella sulla barra del livello dichiarato
        for i, r in enumerate(result_rows):
            dec_j = LAMF_KEYS.index(r['declared'])
            bar_x = i + offsets[dec_j]
            bar_h = r[r['declared']]
            ax.text(bar_x, bar_h + 1.5, '★',
                    ha='center', fontsize=10, color='#1f2937')

        act_labels = [r['name'] for r in result_rows]
        ax.set_xticks(x)
        ax.set_xticklabels(act_labels, rotation=15, ha='right', fontsize=9)
        ax.set_ylabel('Mean article %', fontsize=10)
        ax.set_ylim(0, 110)
        ax.set_title(
            'Classifier output vs. declared Lamfalussy level (financial regulation ground truth)',
            fontsize=11, pad=14
        )
        ax.grid(axis='y', alpha=0.3)

        star_patch = mpatches.Patch(color='none', label='★ = declared level')
        handles, labels_leg = ax.get_legend_handles_labels()
        ax.legend(handles + [star_patch], labels_leg + ['★ = declared level'],
                  title='Legend', loc='upper right', fontsize=8)

        plt.tight_layout()
        png_path = os.path.join(output_path, 'ground_truth_comparison.png')
        plt.savefig(png_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'\nGrafico salvato: {png_path}')


| Atto | Declared | Predicted | H | L1% | L2% | L3% | L4% | Match |
|---|---|---|---|---|---|---|---|---|
| MiFID II | L1 | L1 | 0.797 | 43.9 | 34.0 | 1.1 | 21.0 | ✓ |
| MiFIR | L1 | L1 | 0.783 | 42.8 | 37.6 | 0.8 | 18.8 | ✓ |
| MiFID II Delegated Regulation 2017/565 | L2 | L2 | 0.632 | 35.8 | 57.9 | 0.5 | 5.8 | ✓ |


Grafico salvato: ..\data\output\ground_truth_validation\ground_truth_comparison.png


C:\Users\claud\AppData\Local\Temp\ipykernel_24536\1537850541.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
